# 02. NDVI (MODIS) i mapa pól (ESA WorldCover)

Oba z Microsoft Planetary Computer, **bez konta**. Linki do plików podpisujemy anonimowo.
`pip install rasterio matplotlib`

In [ ]:
import requests
import numpy as np
import rasterio
import matplotlib.pyplot as plt

STAC = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
SIGN = "https://planetarycomputer.microsoft.com/api/sas/v1/sign"
POLAND = [14.0, 49.0, 24.2, 55.0]


def search(collection, date, bbox=POLAND):
    body = {"collections": [collection], "bbox": bbox, "datetime": date, "limit": 50}
    return requests.post(STAC, json=body).json()["features"]


def signed(href):
    return requests.get(SIGN, params={"href": href}).json()["href"]

## NDVI MODIS

Ile kafelków pokrywa Polskę w jednym terminie i jakie warstwy ma każdy?

In [ ]:
# MODIS NDVI: jeden termin (koniec czerwca 2018, susza)
items = [i for i in search("modis-13Q1-061", "2018-06-20/2018-06-30") if i["id"].startswith("MOD13Q1")]
for item in items:
    print(item["id"], item["properties"]["start_datetime"][:10])
print(list(items[0]["assets"]))

In [ ]:
item = next(i for i in items if ".h19v03." in i["id"])
with rasterio.open(signed(item["assets"]["250m_16_days_NDVI"]["href"])) as src:
    print("rozmiar:", src.width, "x", src.height, "| typ:", src.dtypes[0], "| nodata:", src.nodata)
    print("układ:", src.crs)
    print("skala:", src.scales[0], "| opis:", src.tags(1))
    ndvi_raw = src.read(1, out_shape=(src.height // 4, src.width // 4))
with rasterio.open(signed(item["assets"]["250m_16_days_pixel_reliability"]["href"])) as src:
    reliability = src.read(1)[::4, ::4]

Surowe wartości / 10 000 = NDVI (-3000 = brak). Jakość piksela: 0 dobra, 1 średnia, 2 śnieg, 3 chmura, 255 brak danych. Do obliczeń bierzemy tylko 0 i 1.

In [ ]:
ndvi = np.where(ndvi_raw == -3000, np.nan, ndvi_raw * 0.0001)
values, counts = np.unique(reliability, return_counts=True)
print("jakość piksela:", dict(zip(values.tolist(), (counts / counts.sum()).round(3).tolist())))

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(ndvi, cmap="RdYlGn", vmin=0, vmax=1)
ax[0].set_title("NDVI, kafelek h19v03 (Polska centralna i północ)")
ax[1].imshow(np.where(reliability == 255, -1, reliability), cmap="tab10", vmin=-1, vmax=8)
ax[1].set_title("jakość piksela")
plt.show()

## Mapa pól (ESA WorldCover, 10 m)

Klasa 40 = pola uprawne. Sprawdzamy, jaka część jednego kafelka to pola.

In [ ]:
# WorldCover: kafelek z Wielkopolska
wc_item = search("esa-worldcover", "2021-01-01/2021-12-31", bbox=[16.9, 52.3, 17.0, 52.4])[0]
with rasterio.open(signed(wc_item["assets"]["map"]["href"])) as src:
    print(wc_item["id"], "|", src.width, "x", src.height, "|", src.crs)
    landcover = src.read(1, out_shape=(src.height // 20, src.width // 20))

classes = {10: "drzewa", 20: "krzewy", 30: "trawy", 40: "pola", 50: "zabudowa", 60: "goła ziemia", 80: "woda", 90: "mokradła"}
values, counts = np.unique(landcover, return_counts=True)
{classes.get(v, v): round(c / counts.sum(), 3) for v, c in zip(values.tolist(), counts.tolist())}

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(landcover == 40, cmap="Greens")
plt.title("pola uprawne (klasa 40)")
plt.show()